This notebook is made to walkthrough how I clean the data in the manager_salary_data.sc file using the Python Pandas library.

In [ ]:
import pandas as pd

data = pd.read_csv('/Users/arthurcosta17/Projects/Manager salaries/data/manager_salary_data.csv')


First we read the csv file with Pandas. I will rename the columns so that the names aren't so long and are easier to query on SQL.

In [321]:
print(data.head())

            Timestamp How old are you?  What industry do you work in?  \
0  4/27/2021 11:02:10            25-34   Education (Higher Education)   
1  4/27/2021 11:02:22            25-34              Computing or Tech   
2  4/27/2021 11:02:38            25-34  Accounting, Banking & Finance   
3  4/27/2021 11:02:41            25-34                     Nonprofits   
4  4/27/2021 11:02:42            25-34  Accounting, Banking & Finance   

                                  Job title  \
0        Research and Instruction Librarian   
1  Change & Internal Communications Manager   
2                      Marketing Specialist   
3                           Program Manager   
4                        Accounting Manager   

  If your job title needs additional context, please clarify here:  \
0                                                NaN                 
1                                                NaN                 
2                                                NaN                

In [322]:


#print(data.columns) # Read off of this to choose following names.

change_list = ['timestamp','age_group','industry','job_title','job_title_context','annual_salary','extra_annual_salary','currency',
'other_currency','income_context','country','usa_state','city','total_work_experience','related_work_experience','highest_education',
'gender','race']

name_change_dict = {}

for index in range(len(change_list)):
    name_change_dict[data.columns[index]] = change_list[index]

data.rename(columns = name_change_dict,inplace=True) # inplace is altering the data DataFrame, instead of making a new one.
print(data.head())

            timestamp age_group                       industry  \
0  4/27/2021 11:02:10     25-34   Education (Higher Education)   
1  4/27/2021 11:02:22     25-34              Computing or Tech   
2  4/27/2021 11:02:38     25-34  Accounting, Banking & Finance   
3  4/27/2021 11:02:41     25-34                     Nonprofits   
4  4/27/2021 11:02:42     25-34  Accounting, Banking & Finance   

                                  job_title job_title_context annual_salary  \
0        Research and Instruction Librarian               NaN        55,000   
1  Change & Internal Communications Manager               NaN        54,600   
2                      Marketing Specialist               NaN        34,000   
3                           Program Manager               NaN        62,000   
4                        Accounting Manager               NaN        60,000   

   extra_annual_salary currency other_currency income_context         country  \
0                  0.0      USD            NaN 

The names are a lot easier to query now. What else do we need to do? There's a lot of diffrent names people use for the same thing. Like 'USA' 'usa' 'us' 'US' 'United States' 'United States of America', these all refer to the same country, USA, but there are so may different names. 

There are some entries we can get rid of, for example, in the columns for currency and other_currency. One is able to input 'other' into the currency column, and then enter a custom currency. Some entries have entries in both, even if the currency column has 'other' filled, for example:

In [323]:
print(data[['currency','other_currency']].loc[776])

currency                                                        USD
other_currency    I work for an online state university, managin...
Name: 776, dtype: str


The entry in 'other_currency' can be disregarded here. If the first entry isn't 'other', then we can ignore any entry in 'other_currency'. We can clean this column a bit then, in case we ever need to count non-null entries inside of it.

In [324]:
data.loc[data['currency'].str.lower() != 'other', 'other_currency'] = None

print(data[['currency','other_currency']].loc[776]) # If it doesn't have NaN or None in 'other_currency' then this cleaning has failed.


currency          USD
other_currency    NaN
Name: 776, dtype: str


It takes some time to run this query (0.9s), but it helps clean the data a bit. The row used as an exmaple before got cleaned up. Unfortunately some people (at least 1) entered their response for the 'income_context' into the 'other_currency' column, so that response got deleted, but this won't be a significant amount of entries so it is ok.

In [325]:
print(data[['currency','other_currency']])

print(data[['currency','other_currency']].loc[1311])

      currency other_currency
0          USD            NaN
1          GBP            NaN
2          USD            NaN
3          USD            NaN
4          USD            NaN
...        ...            ...
28240      USD            NaN
28241      USD            NaN
28242      USD            NaN
28243      USD            NaN
28244      USD            NaN

[28245 rows x 2 columns]
currency          Other
other_currency      MYR
Name: 1311, dtype: str


One way to get around the USA vs usa vs us vs... issue is to look if the 'usa_state' entry. If it has an entry, which isn't na, then we can autofill the country column to be USA, so that the entries are consistent.

In [326]:
data.loc[ data['usa_state'].notna(), 'country' ] = 'USA' # If a state was inputted, then we must be in the USA

usa_like = ['usa','us','unitedstates','unitedstatesofamerica']

data.loc[ data['country'].str.lower().str.replace(' ','').isin(usa_like), 'country' ] = 'USA' # For entries who did not input a state.

print(data[['country','usa_state']].head())

          country       usa_state
0             USA   Massachusetts
1  United Kingdom             NaN
2             USA       Tennessee
3             USA       Wisconsin
4             USA  South Carolina


There is another issue, with UK vs United Kingdom. This isn't as annoying but it is also an easy fix.

In [327]:
uk_like = ['unitedkingdom','uk','britain']

data.loc[ data['country'].str.lower().str.replace(' ','').isin(uk_like) , 'country'] = 'UK'

print(data[['country','usa_state']].head(25))

   country       usa_state
0      USA   Massachusetts
1       UK             NaN
2      USA       Tennessee
3      USA       Wisconsin
4      USA  South Carolina
5      USA   New Hampshire
6      USA  South Carolina
7      USA         Arizona
8      USA        Missouri
9      USA         Florida
10     USA             NaN
11     USA    Pennsylvania
12     USA        Michigan
13     USA       Minnesota
14  Canada             NaN
15      UK             NaN
16     USA        Illinois
17     USA      California
18     USA         Georgia
19     USA         Florida
20     USA    Pennsylvania
21     USA             NaN
22  Canada             NaN
23     USA            Ohio
24     USA         Florida


In [328]:
print(data[['extra_annual_salary']].loc[range(220,230)])

     extra_annual_salary
220                  0.0
221                  NaN
222                  NaN
223                  0.0
224                  0.0
225                  NaN
226                  NaN
227                  NaN
228                  NaN
229               2000.0


Can see that in this column, there are NaN entries, but also 0 as entries. Do we prefer NaN or 0? Well NaN has the benefit of it not being added up, say for example we want to take an average of people's extra_salary, then we wouldn't have to filter by saying HAVING extra_annual_salary != 0. But if we keep 0 it might less be more flexible with what data we can calculate, instead of having NaN instead. It is just something we have to be aware of.

In [329]:
data.loc[ data['extra_annual_salary'].isna(), 'extra_annual_salary'] = 0
print(data[['extra_annual_salary']].loc[range(220,230)])

     extra_annual_salary
220                  0.0
221                  0.0
222                  0.0
223                  0.0
224                  0.0
225                  0.0
226                  0.0
227                  0.0
228                  0.0
229               2000.0


Another issue is the commas, almost all salary entries have a comma in them. Have to remove all these commas so that this is a valid numeric data type. To do this I can replace all commas with ''.

In [330]:
print(data[['annual_salary','extra_annual_salary']])
print('Changes to:')
data['annual_salary'] = data['annual_salary'].str.replace(',','')
#data['extra_annual_salary'].str.replace(',','') this Series is already properly formatted.
print(data[['annual_salary',]])

      annual_salary  extra_annual_salary
0            55,000                  0.0
1            54,600               4000.0
2            34,000                  0.0
3            62,000               3000.0
4            60,000               7000.0
...             ...                  ...
28240         35000                  0.0
28241             0                  0.0
28242         31200                  0.0
28243         37650                  0.0
28244         35000                  0.0

[28245 rows x 2 columns]
Changes to:
      annual_salary
0             55000
1             54600
2             34000
3             62000
4             60000
...             ...
28240         35000
28241             0
28242         31200
28243         37650
28244         35000

[28245 rows x 1 columns]


I think this has been cleaned enough now. At least it is clean enough to be usable, in my eyes at least.

In [ ]:
#data.to_csv('manager_salary_data_cleaned.csv', index=True)

There are some thing I haven't been able to fix properly. For example, there's an entry which is 'United Kingdom '. There's a space at the end, so the way I corrected it earlier won't work, because techincally 'united kingdom'. Actually no I was, first i tried ''.join(... .str.split('')) but ChatGPT helped to come up with .str.replace(' ','') instead.

Anyways this programme cleaned the data quite well in my eyes, so now I'll try to run some queries on it using Docker and PostgreSQL